# FinRisk Radar — Notebook 02: ML Training

Step-by-step walkthrough of the two-stage ML pipeline.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import roc_curve, auc
plt.style.use('dark_background')

## Stage 1: Isolation Forest (Unsupervised)

In [ ]:
from ml.train import load_features, train_isolation_forest

df, X, y = load_features()
print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Distress rate: {y.mean():.1%}')

iso_model, if_scores = train_isolation_forest(X)
print(f'\nIsolation Forest anomaly scores:')
print(f'  Min: {if_scores.min():.3f}')
print(f'  Max: {if_scores.max():.3f}')
print(f'  Mean (safe):      {if_scores[y==0].mean():.3f}')
print(f'  Mean (distressed): {if_scores[y==1].mean():.3f}')

## Stage 2: XGBoost Classifier (Supervised)

In [ ]:
from ml.train import train_xgboost

xgb_pipeline, xgb_proba, metrics = train_xgboost(X, y)
print('XGBoost CV Results:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

## ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y, xgb_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#3b82f6', lw=2.5, label=f'XGBoost (AUROC = {roc_auc:.3f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.4, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Distress Prediction')
ax.legend(loc='lower right')
ax.fill_between(fpr, tpr, alpha=0.1, color='#3b82f6')
plt.tight_layout()
plt.savefig('../data/processed/roc_curve.png', dpi=150)
plt.show()

## SHAP Global Feature Importance

In [ ]:
from ml.feature_engineering import FEATURE_COLS
from ml.shap_explainer import build_explainer

explainer, scaler = build_explainer(xgb_pipeline, X)
X_scaled = scaler.transform(X)
shap_values = explainer.shap_values(X_scaled)

available_feats = [c for c in FEATURE_COLS if c in pd.read_csv('../data/processed/features.csv').columns]

shap.summary_plot(
    shap_values, X_scaled,
    feature_names=available_feats,
    plot_type='bar',
    show=False
)
plt.title('Global Feature Importance (SHAP)')
plt.tight_layout()
plt.savefig('../data/processed/shap_global.png', dpi=150)
plt.show()
print('\nInterpretation: Altman Z-Score and Debt/Equity tend to be the top drivers.')

## Ensemble Risk Score Distribution

In [ ]:
from ml.train import compute_risk_score, risk_label

if_norm = (if_scores - if_scores.min()) / (if_scores.max() - if_scores.min())
risk_scores = compute_risk_score(if_norm, xgb_proba)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#ef4444' if s >= 70 else '#f59e0b' if s >= 40 else '#10b981' for s in risk_scores]

ax.hist(risk_scores, bins=30, color='#3b82f6', edgecolor='none', alpha=0.7)
ax.axvline(40, color='#f59e0b', linestyle='--', label='Medium threshold (40)')
ax.axvline(70, color='#ef4444', linestyle='--', label='High threshold (70)')
ax.set_xlabel('Risk Score (0–100)')
ax.set_ylabel('Count')
ax.set_title('Risk Score Distribution')
ax.legend()
plt.tight_layout()
plt.show()

labels = pd.Series([risk_label(s) for s in risk_scores])
print(labels.value_counts())